<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L05-clause-classification-abstention/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L05-clause-classification-abstention/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/document-intelligence/lessons/P02-L05-clause-classification-abstention/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have. The cell below installs `tokenizers==0.23.2`, and does
nothing where they are already present. On Kaggle, switch Internet on in the notebook's
settings first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("tokenizers", "tokenizers==0.23.2")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/document-intelligence/lessons/P02-L05-clause-classification-abstention/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P02-L05 · Contract clause classification and calibrated abstention

**You will build:** a contract clause classifier that says "I don't know" at a rate you chose
and priced — a subword vocabulary you prove reproducible, a multinomial logistic regression
in numpy with a stopping rule you can state, probabilities calibrated by temperature scaling,
an abstention threshold fitted where it may be fitted, and a hand-off of every abstained
clause to module 1's review queue.

**Time:** ~80 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download of data,
no model API · **Prerequisites:** T00-L01 (the 8 GB track) and P02-L01 (field extraction you
can measure), whose record format, scorer and review queue this lesson reuses.

By the end you will be able to:

1. Implement a bag-of-subwords featuriser over a byte-pair-encoding vocabulary fitted with
   `tokenizers` on the training split only, and verify that the vocabulary is byte-identical
   from one run to the next.
2. Implement multinomial logistic regression in numpy — softmax, the cross-entropy gradient
   and a gradient-descent loop with a stopping rule you can state.
3. Measure per-clause precision, recall and F1 with abstentions counted, and the recall that
   abstaining costs each clause type.
4. Implement temperature scaling and the expected calibration error, fitted on a calibration
   split, and measure what calibration changes and what it cannot.
5. Choose an abstention threshold from the coverage/accuracy curve, explain why it must not
   be tuned on the split you report, and price the abstention rate it implies.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import hashlib
import io
import json
import math
import random
import sys
import time
import traceback
from collections import Counter
from typing import Callable, Iterable, Mapping, NamedTuple, Sequence

import numpy as np
import tokenizers
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· tokenizers", tokenizers.__version__, "· matplotlib", matplotlib.__version__)
print("no contract library, no model API: a clause classifier you build, calibrate and price")
print("from a synthetic corpus the notebook writes for itself.\n")

# The six clause types this programme classifies, in a fixed order: a label's index in this
# tuple is its class number everywhere below.
LABELS = ("governing_law", "limitation_of_liability", "termination_for_convenience",
          "assignment", "indemnity", "change_of_control")
# The two pairs a reviewer confuses, and so does a model: they share their vocabulary.
CONFUSABLE_PAIRS = (("assignment", "change_of_control"),
                    ("indemnity", "limitation_of_liability"))
ABSTAIN = -1            # the prediction a classifier makes when it says "I don't know"

# Four splits, four jobs. Nothing is ever tuned on a split whose job is to be reported.
SPLITS = ("train", "cal", "test", "next_quarter")
SPLIT_SIZES = {"train": 1800, "cal": 600, "test": 600, "next_quarter": 3000}
SPLIT_JOBS = {
    "train": "fit the vocabulary and the weights",
    "cal": "fit the temperature and the abstention threshold",
    "test": "report what the policy does, touched once",
    "next_quarter": "stand in for the future, so this lesson can check the report",
}
CORPUS_SEED = 20260923

VOCAB_SIZE = 800        # subwords, including the one special token
LEARNING_RATE = 4.0     # full-batch gradient descent on L2-normalised features
L2 = 1e-3               # ridge penalty on the weights, never on the bias
TOL = 1e-5              # stop once one step lowers the training loss by less than this
MAX_ITER = 3000
N_BINS = 15             # reliability bins for the expected calibration error
T_BOUNDS = (0.05, 20.0) # the temperature is searched inside this interval

# The accuracy you promise on the clauses the classifier answers. It is YOURS to choose;
# section 11 prices the choice. Change it, re-run from section 9 down, and read the bill.
TARGET_ACCURACY = 0.98


class Model(NamedTuple):
    """A trained multinomial logistic regression and the record of how it was trained."""
    W: np.ndarray           # (n_features, n_classes)
    b: np.ndarray           # (n_classes,)
    n_iter: int             # gradient steps taken
    losses: list            # training loss before the first step and after every step


class ClassScore(NamedTuple):
    """What one clause type scored, in module 1's vocabulary of counts and ratios."""
    label: str
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float


class Policy(NamedTuple):
    """An abstention policy: divide the logits by `temperature`, answer iff the top
    calibrated probability is at least `threshold`."""
    temperature: float
    threshold: float
    target_accuracy: float


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("bag_of_subwords",),
    "exercise 2": ("softmax", "loss_and_gradient"),
    "exercise 3": ("train_softmax_regression",),
    "exercise 4": ("score_classes", "macro_f1"),
    "exercise 5": ("fit_temperature", "expected_calibration_error"),
    "exercise 6": ("coverage_accuracy_curve", "choose_threshold", "predict_or_abstain"),
    "exercise 7": ("fit_policy", "to_review_records"),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 1"] -> "exercise 1 (bag_of_subwords)"; several -> "exercises 2, 3 and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in a notebook; close it quietly in a script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)

## 1. The corpus, and the two pairs that make it hard

There is no contract library in this environment and no annotated contract set on a
required path, so `build_corpus()` drafts clauses from a fixed seed: six clause types, each
with its own stock sentences, written with the varied parties, jurisdictions, caps and notice
periods a real contract stack has. Three things make it realistic rather than easy:

* **Hybrid clauses.** Some assignment clauses say a change of control counts as an
  assignment; some change-of-control clauses require consent to transfer; some liability
  caps carve out the indemnity. Each is labelled by its primary purpose, as a reviewer
  would label it, and each borrows its neighbour's vocabulary.
* **Smudged scans.** Some clauses carry OCR substitutions — `thls`, `porty`, `right5`.
* **Truncation.** Some clauses were cut short by an upstream segmenter.

The confusion is not an artefact of this generator. Five of the six types are label
categories in CUAD, an expert-annotated contract-review dataset (arXiv 2103.06268), and
CUAD's own definition of change of control already includes assignment by operation of law.
Each clause records which of the three happened to it, so later cells can count where the
errors and the abstentions land rather than assert it. Run it and read one clause per type.

In [ ]:
_PRIORS = (0.18, 0.20, 0.16, 0.17, 0.18, 0.11)
_PARTNER = {"assignment": "change_of_control", "change_of_control": "assignment",
            "indemnity": "limitation_of_liability", "limitation_of_liability": "indemnity",
            "termination_for_convenience": "change_of_control", "governing_law": "indemnity"}
_PARTIES = ("the Supplier", "the Customer", "the Provider", "the Client", "the Licensor",
            "the Licensee")
_JURISDICTIONS = ("England and Wales", "the Netherlands", "Ireland", "the State of New York",
                  "Singapore", "Germany", "Scotland", "Sweden")
_CAPS = ("the total fees paid in the twelve months preceding the event giving rise to the claim",
         "EUR 1,000,000", "125% of the annual charges", "the Charges paid under this Agreement",
         "GBP 500,000 in aggregate")
_DAYS = ("30", "60", "90", "thirty", "sixty", "ninety")
_LEADS = ("", "", "", "Subject to clause {k}, ", "Save as expressly provided otherwise, ",
          "For the avoidance of doubt, ", "Notwithstanding any other provision of this Agreement, ")
_CORE = {
    "governing_law": (
        "this Agreement and any dispute or claim arising out of or in connection with it shall be governed by and construed in accordance with the laws of {j}.",
        "the courts of {j} shall have exclusive jurisdiction to settle any dispute or claim arising out of or in connection with this Agreement.",
        "this Agreement is governed by the law of {j}, and each party irrevocably submits to the jurisdiction of its courts.",
        "any non-contractual obligations arising out of or in connection with this Agreement shall be governed by the laws of {j}.",
        "the United Nations Convention on Contracts for the International Sale of Goods shall not apply to this Agreement.",
    ),
    "limitation_of_liability": (
        "neither party's total aggregate liability arising under or in connection with this Agreement, whether in contract, tort (including negligence) or otherwise, shall exceed {cap}.",
        "neither party shall be liable for any indirect or consequential loss, loss of profits, loss of revenue or loss of goodwill.",
        "nothing in this Agreement limits or excludes liability for death or personal injury caused by negligence, or for fraud or fraudulent misrepresentation.",
        "{p} shall have no liability for any loss or damage to the extent caused by the acts or omissions of the other party.",
        "the maximum liability of {p} in respect of all claims in any contract year shall be limited to {cap}.",
    ),
    "termination_for_convenience": (
        "{p} may terminate this Agreement for convenience at any time by giving not less than {d} days' written notice to the other party.",
        "either party may terminate this Agreement without cause on {d} days' prior written notice.",
        "{p} may end this Agreement for any reason or no reason by written notice, and termination shall take effect {d} days after the date of the notice.",
        "on termination for convenience {p} shall pay for all Services performed up to the effective date of termination.",
        "no termination fee or other compensation shall be payable where this Agreement is terminated under this clause.",
    ),
    "assignment": (
        "neither party shall assign, transfer, charge, sub-contract or deal in any other manner with any of its rights or obligations under this Agreement without the prior written consent of the other party.",
        "{p} may assign or novate this Agreement to any of its Affiliates without the consent of the other party.",
        "such consent shall not be unreasonably withheld, conditioned or delayed.",
        "{p} may assign its rights under this Agreement to a successor in connection with a merger, reorganisation or sale of all or substantially all of its assets.",
        "any purported assignment in breach of this clause shall be void.",
    ),
    "indemnity": (
        "{p} shall indemnify, defend and hold harmless the other party against all losses, liabilities, damages, costs and expenses arising out of any third-party claim.",
        "{p} shall indemnify the other party in full against any claim that the Services infringe the intellectual property rights of a third party.",
        "the indemnified party shall notify {p} promptly of any claim and shall allow {p} the conduct of the defence and settlement of that claim.",
        "{p} shall keep the other party indemnified against all costs, including reasonable legal fees, incurred as a result of any breach of the data protection obligations.",
        "the indemnified party shall take all reasonable steps to mitigate any loss in respect of which it claims under this indemnity.",
    ),
    "change_of_control": (
        "if {p} undergoes a Change of Control, the other party may terminate this Agreement by written notice within {d} days of becoming aware of it.",
        "Change of Control means the acquisition by any person of more than fifty per cent of the voting shares of {p}, or of the power to direct its management, whether by merger, sale of shares or otherwise.",
        "{p} shall notify the other party in writing promptly after any Change of Control takes effect.",
        "a change in the ownership or control of {p} shall require the prior written consent of the other party.",
        "any person who acquires control of {p} shall be bound by the obligations of {p} under this Agreement.",
    ),
}
# Sentences written for one clause type in its neighbour's vocabulary: what makes a hybrid.
_BRIDGE = {
    "assignment": (
        "for the purposes of this clause, a change of control of {p} shall be treated as an assignment of this Agreement.",
        "an assignment by operation of law following a merger or acquisition of {p} requires the consent of the other party."),
    "change_of_control": (
        "any transfer of this Agreement to the acquirer following a change of control shall require the consent of the other party.",
        "{p} may not assign this Agreement to the acquiring entity without the prior written consent of the other party."),
    "indemnity": (
        "the liability of {p} under this indemnity shall not exceed {cap}.",
        "the indemnity in this clause is not subject to the limitations of liability elsewhere in this Agreement."),
    "limitation_of_liability": (
        "the cap in this clause shall not apply to the obligations of {p} under the indemnity for third-party claims.",
        "the exclusions of liability in this clause shall not limit any claim for losses, damages or costs under an indemnity."),
    "termination_for_convenience": (
        "{p} may terminate this Agreement on written notice following any change of control of the other party.",),
    "governing_law": (
        "any claim for losses or damages arising out of this Agreement shall be brought only in the courts of {j}.",),
}
_SMUDGE = {"l": "1", "i": "l", "o": "0", "e": "c", "m": "rn", "n": "ri", "a": "o", "s": "5"}


def _fill(template: str, rng: random.Random) -> str:
    return template.format(p=rng.choice(_PARTIES), j=rng.choice(_JURISDICTIONS),
                           cap=rng.choice(_CAPS), d=rng.choice(_DAYS), k=rng.randint(2, 30))


def _draft_clause(label: str, rng: random.Random) -> dict:
    """One clause of type `label`, with a record of what was done to it."""
    core = _CORE[label]
    sentences = [_fill(s, rng) for s in rng.sample(core, rng.choice((1, 1, 2, 2, 3)))]
    hybrid = rng.random() < 0.25
    if hybrid:  # one of its own sentences, one of its neighbour's, one bridge between them
        sentences = [_fill(rng.choice(core), rng), _fill(rng.choice(_CORE[_PARTNER[label]]), rng),
                     _fill(rng.choice(_BRIDGE[label]), rng)]
        rng.shuffle(sentences)
        sentences = sentences[:rng.choice((2, 3))]
    lead = _fill(rng.choice(_LEADS), rng)
    body = " ".join(s[0].upper() + s[1:] if i or not lead else s
                    for i, s in enumerate(sentences))
    text = lead + body
    text = text[0].upper() + text[1:]
    words = text.split()
    truncated = rng.random() < 0.12 and len(words) > 12
    if truncated:
        words = words[:max(6, int(len(words) * rng.uniform(0.3, 0.6)))]
    smudged = rng.random() < 0.25
    if smudged:
        for _ in range(rng.randint(1, 4)):
            i = rng.randrange(len(words))
            spots = [c for c, ch in enumerate(words[i]) if ch in _SMUDGE]
            if spots:
                c = rng.choice(spots)
                words[i] = words[i][:c] + _SMUDGE[words[i][c]] + words[i][c + 1:]
    return {"text": " ".join(words), "label": label,
            "hybrid": hybrid, "smudged": smudged, "truncated": truncated}


def build_corpus(sizes: Mapping[str, int] = SPLIT_SIZES, seed: int = CORPUS_SEED) -> dict:
    """Deterministic clause corpus: split name -> list of clauses, each a dict with a
    `clause_id`, its `text`, its gold `label` and what was done to it.

    Boilerplate repeats in real contract stacks, and a repeat that straddles two splits is a
    test clause the model has already read. So the clauses are drafted as ONE pool in which no
    text appears twice (a repeat is redrafted), and only then shuffled and cut into splits —
    which also keeps the splits exchangeable: none of them is systematically easier.
    """
    rng = random.Random(seed)
    pool, seen = [], set()
    for _ in range(sum(sizes.values())):
        label = rng.choices(LABELS, weights=_PRIORS)[0]
        for _attempt in range(1000):
            clause = _draft_clause(label, rng)
            if clause["text"] not in seen:
                break
        else:
            raise RuntimeError(f"could not draft a new {label} clause")
        seen.add(clause["text"])
        pool.append(clause)
    rng.shuffle(pool)
    prefix = {"train": "TR", "cal": "CA", "test": "TE", "next_quarter": "NQ"}
    corpus, start = {}, 0
    for split, n in sizes.items():
        corpus[split] = [{"clause_id": f"{prefix[split]}-{i:04d}", **c}
                         for i, c in enumerate(pool[start:start + n])]
        start += n
    return corpus


CORPUS = build_corpus()
Y = {s: np.array([LABELS.index(c["label"]) for c in CORPUS[s]]) for s in SPLITS}
print(f"{'clause type':30s}" + "".join(f"{s:>14s}" for s in SPLITS))
for _k, _lab in enumerate(LABELS):
    print(f"{_lab:30s}" + "".join(f"{int((Y[s] == _k).sum()):14d}" for s in SPLITS))
print(f"{'total':30s}" + "".join(f"{len(CORPUS[s]):14d}" for s in SPLITS))
print(f"\nhybrid clauses in train: {sum(c['hybrid'] for c in CORPUS['train'])}, smudged: "
      f"{sum(c['smudged'] for c in CORPUS['train'])}, truncated: "
      f"{sum(c['truncated'] for c in CORPUS['train'])}\n")
for _lab in LABELS:
    _c = next(c for c in CORPUS["train"] if c["label"] == _lab)
    print(f"[{_lab}] {_c['clause_id']}\n  {_c['text']}\n")

## 2. A subword vocabulary, and proof that it is the same vocabulary every time

The classifier will read a clause as a bag of **subwords**. `tokenizers` learns a byte-pair
encoding vocabulary: it starts from characters and keeps merging the most frequent adjacent
pair until it has `VOCAB_SIZE` entries. This is the byte-pair scheme Sennrich, Haddow and
Birch introduced for encoding rare and unknown words as sequences of subword units (arXiv
1508.07909): a smudged word the training clauses never contained still reads as known
subwords rather than `[UNK]`, where a whole-word vocabulary would have nothing for it. Known
is not the same as useful: the cell prints the smudge's subwords beside those of the clean
word it came from, so you can see how much, if anything, the two share.

Two disciplines, both enforced below. The vocabulary is fitted on the **training split
only** — it is a learned component, and a vocabulary that has seen the test clauses has
seen the test. And it must be **reproducible**: this programme's module map flagged this
module as carrying real implementation risk in this package set, so the first thing done
with it is to train it twice, the second time on the clauses in reverse order, and compare
a fingerprint of the vocabulary and merge list byte for byte. The same fingerprint printed
on Python 3.11 and 3.12 is how this course checks it across interpreters.

In [ ]:
def train_subword_vocab(texts: Sequence[str], vocab_size: int = VOCAB_SIZE) -> Tokenizer:
    """A byte-pair-encoding vocabulary over `texts`: lower-cased, split on whitespace and
    punctuation, merges kept only for pairs seen at least twice, `[UNK]` as id 0."""
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.Lowercase()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
                                  special_tokens=["[UNK]"], show_progress=False)
    tok.train_from_iterator(list(texts), trainer)
    return tok


def vocab_fingerprint(tok: Tokenizer) -> str:
    """SHA-256 over the vocabulary in id order and the merge list in merge order."""
    model = json.loads(tok.to_str())["model"]
    vocab = sorted(model["vocab"].items(), key=lambda kv: kv[1])
    payload = json.dumps({"vocab": vocab, "merges": model["merges"]}, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


TRAIN_TEXTS = [c["text"] for c in CORPUS["train"]]
_t0 = time.perf_counter()
TOKENIZER = train_subword_vocab(TRAIN_TEXTS)
_secs = time.perf_counter() - _t0
_again = train_subword_vocab(list(reversed(TRAIN_TEXTS)))
FINGERPRINT = vocab_fingerprint(TOKENIZER)
print(f"vocabulary: {TOKENIZER.get_vocab_size()} subwords from the {len(TRAIN_TEXTS)} training "
      f"clauses only, trained in {_secs:.3f} s")
print(f"fingerprint, corpus order   : {FINGERPRINT[:16]}")
print(f"fingerprint, reversed order : {vocab_fingerprint(_again)[:16]}")
print(f"byte-identical vocabulary and merges: {FINGERPRINT == vocab_fingerprint(_again)}\n")


def _words(text: str) -> list[str]:
    return [w.lower() for w, _ in TOKENIZER.pre_tokenizer.pre_tokenize_str(text)]


_train_words = Counter(w for t in TRAIN_TEXTS for w in _words(t))
_word_vocab = {w for w, _ in _train_words.most_common(VOCAB_SIZE)}
_test_words = [w for c in CORPUS["test"] for w in _words(c["text"])]
_oov = sum(w not in _word_vocab for w in _test_words)
_unk = TOKENIZER.token_to_id("[UNK]")
_enc = TOKENIZER.encode_batch([c["text"] for c in CORPUS["test"]])
_n_sub = sum(len(e.ids) for e in _enc)
_n_unk = sum(e.ids.count(_unk) for e in _enc)
print(f"a whole-word vocabulary of the same size misses {_oov} of {len(_test_words)} test words "
      f"({100 * _oov / len(_test_words):.1f}%)")
print(f"the subword vocabulary maps {_n_unk} of {_n_sub} test subwords to [UNK]")
_example = next(w for w in _test_words if w.isalpha() and w not in _train_words)
print(f"a test word never seen in training, {_example!r}, reads as "
      f"{TOKENIZER.encode(_example).tokens}")


def _clean_form(word: str) -> str | None:
    """The training word a smudge came from, if undoing one substitution recovers it."""
    for bad, good in ((v, k) for k, v in _SMUDGE.items()):
        at = word.find(bad)
        while at >= 0:
            candidate = word[:at] + good + word[at + len(bad):]
            if candidate in _train_words:
                return candidate
            at = word.find(bad, at + 1)
    return None


_clean = _clean_form(_example)
if _clean is None:
    print("  no single substitution turns it back into a training word")
else:
    _shared = set(TOKENIZER.encode(_example).tokens) & set(TOKENIZER.encode(_clean).tokens)
    print(f"  its clean form {_clean!r} reads as {TOKENIZER.encode(_clean).tokens}: "
          f"{len(_shared)} subword(s) in common {sorted(_shared)}")

## 3. Exercise 1 — `bag_of_subwords`

The classifier needs one fixed-length row of numbers per clause. The tokenizer hands you, for
each clause, a list of subword ids — with repeats, because "party" appears three times in a
long clause. The row you build has one column per vocabulary entry. Two choices are the
standard ones for linear text models and both are written into the docstring: damp the
counts with `log1p`, so a word said five times is not five times the evidence; then scale
each row to unit length, so a long clause does not shout over a short one.

<details><summary>💡 Hint 1 — what to think about</summary>

The ids repeat, and a repeat is information. Look hard at what happens in numpy when one
index appears twice in a fancy-indexed `+=`. And a truncated clause can arrive with no ids at
all: decide what its row is before you divide by its length.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the ids against the vocabulary size first. Start from a zero matrix of shape
(number of clauses, vocab_size). For each clause, add one per occurrence of each id with an
accumulating add that respects repeats (numpy has one; a Counter works too). Apply log1p to
the whole matrix, compute each row's Euclidean norm, and divide only the rows whose norm is
above zero.

</details>

In [ ]:
def bag_of_subwords(id_lists: Sequence[Sequence[int]], vocab_size: int) -> np.ndarray:
    """One row per clause: log1p of each subword's count, the row scaled to unit L2 length.

    * Repeats count: an id that appears three times in a clause contributes a count of 3.
    * Counts are damped with ``np.log1p`` BEFORE the row is normalised.
    * Each row is divided by its own Euclidean norm. An empty clause (no ids) stays a row of
      zeros — never ``nan``.
    * An id outside ``[0, vocab_size)`` is a bug upstream: raise ``ValueError``.

    Returns a float64 array of shape ``(len(id_lists), vocab_size)``.

    Example:
        >>> X = bag_of_subwords([[2, 2, 0], []], 3)
        >>> np.round(X, 4).tolist()      # counts (1, 0, 2) -> log1p -> unit length
        [[0.5336, 0.0, 0.8457], [0.0, 0.0, 0.0]]
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_bag() -> None:
    X = bag_of_subwords([[1, 1, 3], [], [2]], 4)
    assert X.shape == (3, 4), f"one row per clause, one column per subword: got {X.shape}"
    want = np.log1p(np.array([0.0, 2.0, 0.0, 1.0]))
    want /= np.linalg.norm(want)
    assert np.allclose(X[0], want), (
        f"row 0 should be {np.round(want, 4).tolist()}, got {np.round(X[0], 4).tolist()} — if "
        "column 1 equals column 3, the repeated id 1 was counted once: X[i, ids] += 1 drops "
        "repeats; use np.add.at or a Counter. If the ratio is 2:1, you skipped log1p")
    assert not np.isnan(X).any() and np.all(X[1] == 0), \
        "an empty clause is a row of zeros — guard the division by a zero norm, no nan"
    assert np.allclose(np.linalg.norm(X[[0, 2]], axis=1), 1.0), \
        "every non-empty ROW has unit length — normalise along axis=1, not the columns"
    try:
        bag_of_subwords([[4]], 4)
        raise AssertionError("id 4 with vocab_size 4 must raise ValueError, not be ignored")
    except ValueError:
        pass
    print("exercise 1 looks right")


_try("exercise 1", _check_bag)

Run this to turn every split into a feature matrix with your function.

In [ ]:
FEATURES: dict[str, np.ndarray] = {}


def _featurise() -> None:
    vocab = TOKENIZER.get_vocab_size()
    for split in SPLITS:
        ids = [e.ids for e in TOKENIZER.encode_batch([c["text"] for c in CORPUS[split]])]
        FEATURES[split] = bag_of_subwords(ids, vocab)
    for split in SPLITS:
        nz = (FEATURES[split] > 0).sum(axis=1)
        print(f"{split:13s} {FEATURES[split].shape[0]:5d} clauses x {FEATURES[split].shape[1]} "
              f"subwords, {nz.mean():.1f} distinct subwords per clause on average")


_try("features", _featurise, needs=("exercise 1",))

## 4. Exercise 2 — `softmax` and `loss_and_gradient`

Multinomial logistic regression scores each clause with one weight vector per clause type,
`Z = X @ W + b`, and turns the six scores into probabilities with the softmax. It is trained
by minimising the mean cross-entropy — minus the log of the probability given to the right
type — plus a ridge penalty `(L2 / 2) * sum(W ** 2)` that keeps the weights small. The bias
is not penalised: it only encodes how common each type is.

The gradient has a famously tidy form. With `P` the probabilities and `Y` the one-hot labels,
the gradient of the mean cross-entropy with respect to `Z` is `(P - Y) / n`, and the chain
rule does the rest. The check compares your gradient with a finite difference, so a sign or
a missing `/ n` cannot hide.

<details><summary>💡 Hint 1 — what to think about</summary>

`exp(1000)` is infinity, and a clause scored 1000 for one type is not unusual after a long
training run. The softmax does not change if you subtract the same number from every score
in a row — which number makes the largest exponent zero? For the loss, ask which parameters
the penalty covers, and whether "mean" means divide by n in the gradient too.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Softmax: subtract each row's maximum, exponentiate, divide by the row sum. Loss: compute P,
take minus the log of each row's entry at its true label, average over rows, add half of L2
times the sum of squared weights. Gradient: copy P, subtract one at each row's true label,
divide by n. The weight gradient is X transposed times that, plus L2 times W. The bias
gradient is that same matrix summed down its columns (the column MEAN of P minus the one-hot
labels, before any division by n): dividing by n twice is the trap. No penalty term.

</details>

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """Row-wise softmax of an ``(n, k)`` array, numerically stable for any finite input.

    Every row of the result is non-negative and sums to 1. A row of huge scores such as
    ``[1000.0, 0.0]`` must give ``[1.0, 0.0]``, not ``nan``.

    Example:
        >>> np.round(softmax(np.array([[0.0, np.log(3.0)], [1000.0, 0.0]])), 4).tolist()
        [[0.25, 0.75], [1.0, 0.0]]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def loss_and_gradient(W: np.ndarray, b: np.ndarray, X: np.ndarray, y: np.ndarray,
                      l2: float) -> tuple[float, np.ndarray, np.ndarray]:
    """Mean cross-entropy plus ``(l2 / 2) * sum(W ** 2)``, and its gradients.

    * ``P = softmax(X @ W + b)``; the loss is ``mean(-log P[i, y[i]]) + (l2 / 2) * sum(W**2)``.
    * ``grad_W = X.T @ (P - Y) / n + l2 * W`` and ``grad_b = mean(P - Y, axis=0)``, with ``Y``
      the one-hot labels. The bias is never penalised.

    Returns ``(loss, grad_W, grad_b)`` with ``grad_W`` shaped like ``W`` and ``grad_b`` like
    ``b``.

    Example:
        >>> X = np.array([[1.0, 0.0], [0.0, 1.0]]); y = np.array([0, 1])
        >>> loss, gW, gb = loss_and_gradient(np.zeros((2, 3)), np.zeros(3), X, y, 0.0)
        >>> round(loss, 4), np.round(gb, 4).tolist()      # uniform over 3 types: log 3
        (1.0986, [-0.1667, -0.1667, 0.3333])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_softmax_and_loss() -> None:
    p = softmax(np.array([[1000.0, 0.0, -1000.0], [1.0, 2.0, 3.0]]))
    assert np.isfinite(p).all(), \
        "softmax overflowed on a score of 1000 — subtract each row's max before exp"
    assert np.allclose(p.sum(axis=1), 1.0), \
        "each ROW must sum to 1 — normalise along axis=1, not axis=0"
    assert np.allclose(p[1], softmax(np.array([[11.0, 12.0, 13.0]]))[0]), \
        "adding a constant to a row must not change its softmax"
    rng = np.random.default_rng(0)
    X, y = rng.normal(size=(7, 4)), np.array([0, 2, 1, 2, 0, 1, 1])
    W, b, l2 = rng.normal(size=(4, 3)), rng.normal(size=3), 0.3
    loss, gW, gb = loss_and_gradient(W, b, X, y, l2)
    assert gW.shape == W.shape and gb.shape == b.shape, "gradients must be shaped like W and b"
    h = 1e-6
    for (i, j) in ((0, 0), (3, 2), (1, 1)):
        Wp, Wm = W.copy(), W.copy()
        Wp[i, j] += h
        Wm[i, j] -= h
        num = (loss_and_gradient(Wp, b, X, y, l2)[0] - loss_and_gradient(Wm, b, X, y, l2)[0]) / (2 * h)
        assert abs(num - gW[i, j]) < 1e-5, (
            f"grad_W[{i},{j}] is {gW[i, j]:.6f} but the loss moves at {num:.6f} — check the "
            "sign of (P - Y), the / n, and that the penalty's gradient is l2 * W for a loss "
            "term of (l2 / 2) * sum(W**2)")
    bp, bm = b.copy(), b.copy()
    bp[1] += h
    bm[1] -= h
    num = (loss_and_gradient(W, bp, X, y, l2)[0] - loss_and_gradient(W, bm, X, y, l2)[0]) / (2 * h)
    assert abs(num - gb[1]) < 1e-5, (
        f"grad_b[1] is {gb[1]:.6f} but the loss moves at {num:.6f} — the bias is NOT "
        "penalised, so its gradient is the column mean of (P - Y) and nothing else")
    print("exercise 2 looks right")


_try("exercise 2", _check_softmax_and_loss)

## 5. Exercise 3 — `train_softmax_regression`, with a stopping rule you can state

Full-batch gradient descent from all-zero weights: every step uses every training clause, so
two runs on the same data take exactly the same path. What remains is the question every
training loop has to answer — when do you stop? — and the answer has to be a rule a reviewer
can read, not "when it looked done". This lesson's rule: **stop after the first step that
lowers the training loss by less than `tol`** (a step that raises it counts, since its
decrease is negative), **or after `max_iter` steps**. You record the loss before the first
step and after every step, so the history proves the rule was followed.

<details><summary>💡 Hint 1 — what to think about</summary>

Three things make or break this: where the weights start, how many losses the history
holds compared with the number of steps, and the exact comparison in the stopping test.
"The decrease" is previous minus new. A version that takes the absolute value, or divides
by the loss, is a different rule.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate lr and max_iter. Start W and b at zeros. Compute the loss and gradients once and
record the loss. Then loop up to max_iter times: step both W and b against their gradients
scaled by lr, recompute loss and gradients at the new point, record the loss, and break if
the previous recorded loss minus this one is below tol. The number of steps is the length of
the history minus one.

</details>

In [ ]:
def train_softmax_regression(X: np.ndarray, y: np.ndarray, n_classes: int,
                             lr: float = LEARNING_RATE, l2: float = L2, tol: float = TOL,
                             max_iter: int = MAX_ITER) -> Model:
    """Full-batch gradient descent on `loss_and_gradient`, from all-zero W and b.

    * ``losses[0]`` is the loss before any step; ``losses[t]`` the loss after step t.
    * Each step: ``W -= lr * grad_W`` and ``b -= lr * grad_b``, both gradients taken at the
      current point, then the new loss is recorded.
    * Stop after the first step with ``losses[t - 1] - losses[t] < tol`` (an increase counts),
      or after ``max_iter`` steps, whichever comes first.
    * ``n_iter`` is the number of steps taken, so ``len(losses) == n_iter + 1``.
    * ``lr <= 0`` or ``max_iter < 1`` is a caller bug: raise ``ValueError``.

    Example:
        >>> X = np.array([[1.0, 0.0], [0.0, 1.0]]); y = np.array([0, 1])
        >>> m = train_softmax_regression(X, y, 2, lr=1.0, l2=0.0, tol=0.05, max_iter=100)
        >>> m.n_iter, len(m.losses), round(m.losses[0], 4)
        (5, 6, 0.6931)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_training() -> None:
    X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.2], [0.1, 1.0]])
    y = np.array([0, 1, 0, 1])
    m = train_softmax_regression(X, y, 2, lr=1.0, l2=0.0, tol=0.05, max_iter=100)
    assert len(m.losses) == m.n_iter + 1, (
        f"{len(m.losses)} losses recorded for {m.n_iter} steps — record the loss BEFORE the "
        "first step and after every step, so there is exactly one more loss than steps")
    assert abs(m.losses[0] - math.log(2)) < 1e-9, \
        "losses[0] must be the loss at all-zero weights: log(n_classes) with no penalty"
    drops = [a - b for a, b in zip(m.losses, m.losses[1:])]
    assert drops[-1] < 0.05 and all(d >= 0.05 for d in drops[:-1]), (
        f"the loss drops per step were {[round(d, 4) for d in drops]} with tol=0.05: stop "
        "after the FIRST step whose drop is below tol, not before it and not after it")
    again = train_softmax_regression(X, y, 2, lr=1.0, l2=0.0, tol=0.05, max_iter=100)
    assert np.array_equal(m.W, again.W), "two runs must give identical weights — start at zero"
    capped = train_softmax_regression(X, y, 2, lr=1.0, l2=0.0, tol=0.0, max_iter=3)
    assert capped.n_iter == 3, "with tol=0 the loop must stop at max_iter"
    wild = train_softmax_regression(X, y, 2, lr=500.0, l2=5.0, tol=1e-9, max_iter=50)
    assert wild.n_iter == 1 and wild.losses[1] > wild.losses[0], (
        "a step that RAISES the loss has a negative decrease, which is below tol: stop there. "
        "Taking abs() of the decrease keeps a diverging run going")
    print("exercise 3 looks right")


_try("exercise 3", _check_training)

Now train the real classifier on the training split, and look at the stopping rule's record.

In [ ]:
MODEL = None
LOGITS: dict[str, np.ndarray] = {}
_FOR_MODEL = ("exercise 1", "exercise 2", "exercise 3")


def _train_the_model() -> None:
    global MODEL
    t0 = time.perf_counter()
    MODEL = train_softmax_regression(FEATURES["train"], Y["train"], len(LABELS))
    secs = time.perf_counter() - t0
    for split in SPLITS:
        LOGITS[split] = FEATURES[split] @ MODEL.W + MODEL.b
    last_drop = MODEL.losses[-2] - MODEL.losses[-1]
    print(f"trained in {secs:.2f} s: {MODEL.n_iter} steps, loss {MODEL.losses[0]:.4f} -> "
          f"{MODEL.losses[-1]:.4f}")
    reason = (f"the last step lowered the loss by {last_drop:.2e}, below TOL = {TOL:g}"
              if MODEL.n_iter < MAX_ITER else f"MAX_ITER = {MAX_ITER} reached")
    print(f"stopped because {reason}")
    for split in SPLITS:
        acc = (LOGITS[split].argmax(axis=1) == Y[split]).mean()
        print(f"  accuracy on {split:13s} {acc:.3f}")


_try("training run", _train_the_model, needs=_FOR_MODEL)

## 6. Exercise 4 — per-clause scores that know what an abstention is

Module 1 scored fields; this module scores clause types, with the same counts. A clause
predicted as the wrong type is a false positive for the type it was given **and** a false
negative for its real type. The new case is `ABSTAIN` (`-1`): the classifier said "I don't
know". That claims nothing, so it is a false positive for no type — but the clause's real
type was not found, so it is a false negative for that one. Abstention can only lower recall.

<details><summary>💡 Hint 1 — what to think about</summary>

In numpy, `counts[-1] += 1` is legal and increments the LAST entry. `change_of_control` is the
last label. And resist the tidy-looking move of dropping abstained clauses before scoring:
ask what that does to the denominator of recall. For `macro_f1`, every type counts equally,
however rare.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Reject a length mismatch or a true label outside the label range. For each type k: true
positives are clauses with true k and predicted k; false positives are predicted k with a
different truth; false negatives are true k with any other prediction, ABSTAIN included.
Guard each ratio against a zero denominator, as module 1 did, and build one ClassScore per
type in label order. `macro_f1` is the plain mean of their f1 values.

</details>

In [ ]:
def score_classes(y_true: np.ndarray, y_pred: np.ndarray,
                  labels: Sequence[str] = LABELS) -> tuple[ClassScore, ...]:
    """Precision, recall and F1 for every clause type, in the order of `labels`.

    * ``y_pred == k`` and ``y_true == k`` → a true positive for k.
    * ``y_pred == k`` and ``y_true != k`` → a false positive for k (and a false negative for
      the true type).
    * ``y_pred == ABSTAIN`` → a false negative for the true type and a false positive for
      NONE. Abstained clauses stay in the evaluation.
    * Precision is 0.0 with no predictions of k, recall 0.0 with no true k, F1 0.0 when both
      are 0.0.
    * Different lengths, or a true label outside ``range(len(labels))``: ``ValueError``.

    Example:
        >>> s = score_classes(np.array([0, 0, 1]), np.array([0, ABSTAIN, 0]), ("a", "b"))
        >>> [(c.label, c.tp, c.fp, c.fn) for c in s]
        [('a', 1, 1, 1), ('b', 0, 0, 1)]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def macro_f1(scores: Sequence[ClassScore]) -> float:
    """Unweighted mean of the per-type F1 scores; 0.0 for an empty sequence.

    Unweighted on purpose, as in module 1: change of control is the rarest type here, and it
    is the one a buyer's lawyer asks about first.

    Example:
        >>> macro_f1([ClassScore("a", 1, 0, 0, 1.0, 1.0, 1.0),
        ...           ClassScore("b", 0, 0, 3, 0.0, 0.0, 0.0)])
        0.5
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_scores() -> None:
    labels = ("a", "b", "c")
    s = score_classes(np.array([0, 0, 1, 2, 2]), np.array([0, ABSTAIN, 2, 2, ABSTAIN]), labels)
    got = [(c.tp, c.fp, c.fn) for c in s]
    assert got[2] == (1, 1, 1), (
        f"type 'c' should be tp=1 fp=1 fn=1, got {got[2]} — if fp is 3, every ABSTAIN (-1) "
        "was counted as a prediction of the LAST type: numpy reads index -1 as the last entry")
    assert got[0] == (1, 0, 1), (
        f"type 'a' should be tp=1 fp=0 fn=1, got {got[0]} — an abstained clause is a false "
        "negative for its true type; if fn is 0 you dropped the abstentions before scoring")
    assert got[1] == (0, 0, 1), f"type 'b' should be tp=0 fp=0 fn=1, got {got[1]}"
    assert s[1].precision == 0.0 and s[1].f1 == 0.0, "no predictions of 'b': precision 0.0, not a crash"
    assert abs(s[0].recall - 0.5) < 1e-12 and abs(s[0].precision - 1.0) < 1e-12, \
        "precision = tp / (tp + fp), recall = tp / (tp + fn)"
    m = macro_f1(s)
    assert abs(m - sum(c.f1 for c in s) / 3) < 1e-12, \
        "macro F1 is the plain mean of the per-type F1 values — do not weight by support"
    print("exercise 4 looks right")


_try("exercise 4", _check_scores)

The classifier on the test split, answering every clause: per-type scores, the confusion
matrix, and where its errors come from.

In [ ]:
def _show_full_coverage_scores() -> None:
    pred = LOGITS["test"].argmax(axis=1)
    scores = score_classes(Y["test"], pred)
    print(f"{'clause type':30s}{'tp':>5s}{'fp':>5s}{'fn':>5s}{'prec':>8s}{'recall':>8s}{'f1':>8s}")
    for s in scores:
        print(f"{s.label:30s}{s.tp:5d}{s.fp:5d}{s.fn:5d}{s.precision:8.3f}{s.recall:8.3f}{s.f1:8.3f}")
    print(f"{'MACRO F1':30s}{'':23s}{macro_f1(scores):8.3f}\n")
    cm = np.zeros((len(LABELS), len(LABELS)), dtype=int)
    np.add.at(cm, (Y["test"], pred), 1)
    short = [lab[:6] for lab in LABELS]
    print("confusion matrix (rows: true type, columns: predicted type)")
    print(f"{'':8s}" + "".join(f"{s:>8s}" for s in short))
    for k, row in enumerate(cm):
        print(f"{short[k]:8s}" + "".join(f"{v:8d}" for v in row))
    pair_ids = {frozenset((LABELS.index(a), LABELS.index(b))) for a, b in CONFUSABLE_PAIRS}
    wrong = np.nonzero(pred != Y["test"])[0]
    in_pairs = sum(frozenset((int(Y["test"][i]), int(pred[i]))) in pair_ids for i in wrong)
    hybrid = np.array([c["hybrid"] for c in CORPUS["test"]])
    print(f"\n{len(wrong)} test errors; {in_pairs} of them sit inside the two confusable pairs")
    print(f"error rate on hybrid clauses {(pred != Y['test'])[hybrid].mean():.3f}, on the rest "
          f"{(pred != Y['test'])[~hybrid].mean():.3f}")


_try("full coverage", _show_full_coverage_scores, needs=_FOR_MODEL + ("exercise 4",))

## 7. Exercise 5 — calibration: make 0.9 mean 90%

The classifier's top probability is its confidence. It is **calibrated** when, among clauses
it gives 0.9, about 90% are right — the reading a production document service documents for
its own field confidences, and the one a reviewer will assume. Nothing in training promises
it: here the ridge penalty and the stopping rule both pull probabilities towards uniform,
and Guo et al. (arXiv 1706.04599) found modern neural networks poorly calibrated.
**Temperature scaling**, their recommended fix, uses one number: divide every logit by `T`
before the softmax. `T > 1` softens, `T < 1` sharpens, and because a whole row is divided by
the same positive number, the top type of a clause cannot change — accuracy is untouched. `T`
minimises the mean negative log-likelihood on the **calibration split**: clauses the
weights never saw, and that the report will never be read from.

The **expected calibration error** measures what is left: group clauses into `n_bins` equal
bins of confidence, bin m covering `((m-1)/M, m/M]` as Guo et al. define it, and average
`|accuracy - mean confidence|` over the bins, each weighted by its share of clauses.

<details><summary>💡 Hint 1 — what to think about</summary>

The negative log-likelihood as a function of log T has a single minimum, so a one-dimensional
search you can trust works. Divide the logits, not the probabilities: renormalising p / T
gives back p. For the bins, check where a confidence of exactly 1.0 lands, and where one of
exactly 0.5 lands when there are four bins.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Fit: define the mean NLL of softmax(logits / T) at the true labels, then golden-section
search (or a fine grid followed by a local refinement) over log T between the logs of the two
T_BOUNDS, and return exp of the best point. ECE: confidence is the row max, the prediction
the row argmax. The bin index is the ceiling of confidence times n_bins, minus one, clipped
to at least zero. For each non-empty bin add (its size / n) times |its accuracy - its mean
confidence|.

</details>

In [ ]:
def fit_temperature(logits: np.ndarray, y: np.ndarray,
                    bounds: tuple[float, float] = T_BOUNDS) -> float:
    """The temperature T in `bounds` minimising mean NLL of ``softmax(logits / T)`` at `y`.

    Any method is fine if it lands on the minimiser to within 1e-3 in ``log T``; golden-section
    search on ``log T`` is the suggested one. Empty input, or a length mismatch between the
    rows of `logits` and `y`, raises ``ValueError``.

    Example:
        >>> z = np.array([[4.0, 0.0], [4.0, 0.0], [4.0, 0.0], [0.0, 4.0]])
        >>> round(fit_temperature(z, np.array([0, 0, 1, 1])), 3)   # 3 of 4 right: soften
        3.641
    """
    # YOUR CODE HERE
    raise NotImplementedError


def expected_calibration_error(probs: np.ndarray, y: np.ndarray, n_bins: int = N_BINS) -> float:
    """Weighted mean of |accuracy - mean confidence| over equal-width confidence bins.

    Confidence is each row's maximum probability, the prediction its ``argmax``. Bin m (for m
    = 1..n_bins) holds confidences in ``((m-1)/n_bins, m/n_bins]`` — right-closed, so 1.0 is in
    the last bin and 0.5 is in bin 2 of 4. Each non-empty bin contributes
    ``(its size / n) * |its accuracy - its mean confidence|``; empty bins contribute nothing.

    Example:
        >>> P = np.array([[0.9, 0.1], [0.9, 0.1], [0.9, 0.1], [0.6, 0.4]])
        >>> round(expected_calibration_error(P, np.array([0, 0, 1, 1]), n_bins=4), 3)
        0.325
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_calibration() -> None:
    rng = np.random.default_rng(1)
    true_logits = rng.normal(scale=2.0, size=(4000, 3))
    u = rng.random(4000)[:, None]
    y = (u > np.cumsum(softmax(true_logits), axis=1)).sum(axis=1)
    t_over = fit_temperature(true_logits * 3.0, y)
    assert 2.5 < t_over < 3.6, (
        f"logits three times too sharp should give T near 3, got {t_over:.3f} — if you got "
        "about 1/3, you multiplied the logits by T instead of dividing by it")
    t_under = fit_temperature(true_logits * 0.5, y)
    assert 0.35 < t_under < 0.6, (
        f"logits half as sharp as they should be need T near 0.5, got {t_under:.3f}")
    z = rng.normal(size=(50, 4))
    probs_before, probs_after = softmax(z), softmax(z / t_over)
    assert np.array_equal(probs_before.argmax(1), probs_after.argmax(1)), \
        "dividing a row by one positive T can never change its top type"
    P = np.array([[0.9, 0.1], [0.9, 0.1], [0.9, 0.1], [0.6, 0.4]])
    got = expected_calibration_error(P, np.array([0, 0, 1, 1]), n_bins=4)
    assert abs(got - 0.325) < 1e-9, (
        f"ECE should be 0.325, got {got:.4f} — bin (0.75, 1] holds three clauses (accuracy "
        "2/3, confidence 0.9) and bin (0.5, 0.75] one (accuracy 0, confidence 0.6); weight "
        "each gap by its bin's SHARE of clauses. The unweighted mean of the gaps is 0.4167")
    edge = np.array([[0.5, 0.5], [0.6, 0.4], [0.75, 0.25]])
    got = expected_calibration_error(edge, np.array([1, 0, 0]), n_bins=4)
    assert abs(got - 0.38333333333333) < 1e-9, (
        f"ECE should be 0.3833, got {got:.4f} — bins are right-closed: 0.5 belongs to "
        "(0.25, 0.5] and 0.75 to (0.5, 0.75]. Floor-based bins put each edge one bin high and "
        "give 0.1167")
    print("exercise 5 looks right")


_try("exercise 5", _check_calibration)

Fit the temperature on the calibration split, then measure what it changed and what it
could not change, on the test split and on the next quarter.

In [ ]:
def _show_calibration() -> None:
    temperature = fit_temperature(LOGITS["cal"], Y["cal"])
    lean = "sharpens: the raw model was UNDERconfident" if temperature < 1 else \
        "softens: the raw model was OVERconfident"
    print(f"temperature fitted on the calibration split: T = {temperature:.3f} ({lean})\n")
    print(f"{'split':14s}{'accuracy':>9s}{'argmax moved':>14s}{'mean conf':>11s}{'-> cal':>8s}"
          f"{'ECE raw':>9s}{'-> cal':>8s}")
    for split in ("cal", "test", "next_quarter"):
        raw, cal = softmax(LOGITS[split]), softmax(LOGITS[split] / temperature)
        moved = int((raw.argmax(axis=1) != cal.argmax(axis=1)).sum())
        acc = (cal.argmax(axis=1) == Y[split]).mean()
        print(f"{split:14s}{acc:9.3f}{moved:14d}{raw.max(1).mean():11.3f}{cal.max(1).mean():8.3f}"
              f"{expected_calibration_error(raw, Y[split]):9.3f}"
              f"{expected_calibration_error(cal, Y[split]):8.3f}")
    print()
    raw_t, cal_t = softmax(LOGITS["test"]), softmax(LOGITS["test"] / temperature)
    for name, P in (("raw", raw_t), ("calibrated", cal_t)):
        answered = P.max(axis=1) >= 0.9
        acc = (P.argmax(axis=1) == Y["test"])[answered].mean() if answered.any() else float("nan")
        print(f"'answer when confidence >= 0.9' on {name:10s} probabilities: "
              f"{int(answered.sum()):3d} of {len(answered)} test clauses answered, accuracy "
              f"{acc:.3f}")
    # rounded, so that a one-ulp difference between two builds of numpy cannot turn a tie
    # into an order and change the count
    rc, cc = np.round(raw_t.max(axis=1), 12), np.round(cal_t.max(axis=1), 12)
    iu = np.triu_indices(len(rc), 1)
    flips = int((np.sign(rc[:, None] - rc[None, :])[iu] *
                 np.sign(cc[:, None] - cc[None, :])[iu] < 0).sum())
    print(f"pairs of test clauses whose confidence ORDER the temperature reversed: {flips} of "
          f"{len(iu[0])}")

    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    edges = np.linspace(0, 1, N_BINS + 1)
    for name, P, marker in (("raw", raw_t, "o"), ("calibrated", cal_t, "s")):
        conf, corr = P.max(axis=1), P.argmax(axis=1) == Y["test"]
        idx = np.clip(np.ceil(conf * N_BINS).astype(int) - 1, 0, N_BINS - 1)
        pts = [(conf[idx == m].mean(), corr[idx == m].mean()) for m in range(N_BINS)
               if (idx == m).sum() >= 5]
        ax.plot(*zip(*pts), marker=marker, label=name)
    ax.plot(edges, edges, color="grey", lw=0.8, ls="--", label="perfectly calibrated")
    ax.set_xlabel("mean confidence in bin")
    ax.set_ylabel("accuracy in bin")
    ax.set_title("Reliability on the test split (bins with 5+ clauses)")
    ax.legend(loc="upper left")
    fig.tight_layout()
    _show(fig)


_try("calibration", _show_calibration, needs=_FOR_MODEL + ("exercise 5",))

## 8. Exercise 6 — the coverage/accuracy curve, and a threshold chosen from it

An abstention rule answers a clause **iff its calibrated confidence is at least a threshold**
and says `ABSTAIN` otherwise. Two numbers describe what it does: **coverage**, the share of
clauses it answers, and **selective accuracy**, its accuracy on those. Walk the clauses from
most to least confident and you trace the whole trade-off at once.

Two details decide whether the curve is honest. Clauses with **equal** confidence are
answered or abstained together — no threshold can split them — so the curve has one point
per distinct confidence, not one per clause. And the curve is **not monotone**: one confident
mistake near the top dents it, and it can recover below. The threshold you want is the
**smallest** one whose answered set meets the target, which is the largest coverage that
keeps the promise.

<details><summary>💡 Hint 1 — what to think about</summary>

After sorting by confidence, cumulative sums give every point of the curve in one pass —
but only the LAST clause of each run of equal confidences is a point a threshold can reach.
For the threshold, ask whether stopping at the first point that misses the target could
leave a larger coverage unclaimed further down.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Curve: sort descending by confidence (a stable sort), take cumulative counts of answered and
correct clauses, keep the positions where the next confidence differs (and the last one),
and read threshold, coverage and accuracy there. Threshold: among all curve points whose
accuracy is at least the target (inclusive), return the lowest threshold; if there is none,
return math.inf. Abstain: the row argmax where the row max is at least the threshold,
ABSTAIN elsewhere.

</details>

In [ ]:
def coverage_accuracy_curve(confidence: np.ndarray,
                            correct: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The operating points of "answer iff confidence >= t", one per distinct confidence.

    Returns ``(thresholds, coverage, accuracy)``, three float arrays ordered from the highest
    threshold down: ``coverage[i]`` is the share of clauses with confidence >=
    ``thresholds[i]``, ``accuracy[i]`` the share of those that are correct. Clauses with equal
    confidence always enter together. Empty input or a length mismatch: ``ValueError``.

    Example:
        >>> t, c, a = coverage_accuracy_curve(np.array([0.9, 0.8, 0.8, 0.6]),
        ...                                   np.array([True, True, False, True]))
        >>> t.tolist(), c.tolist(), np.round(a, 3).tolist()
        ([0.9, 0.8, 0.6], [0.25, 0.75, 1.0], [1.0, 0.667, 0.75])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def choose_threshold(confidence: np.ndarray, correct: np.ndarray,
                     target_accuracy: float) -> float:
    """The smallest threshold whose answered clauses reach `target_accuracy` (inclusive).

    Candidates are the distinct confidences themselves. Accuracy is not monotone in the
    threshold, so every candidate is considered, not just the ones above the first miss.
    Returns ``math.inf`` (answer nothing) when no threshold reaches the target. A target
    outside ``(0, 1]`` raises ``ValueError``.

    Example:
        >>> choose_threshold(np.array([0.9, 0.8, 0.8, 0.6]),
        ...                  np.array([True, True, False, True]), 0.75)
        0.6
    """
    # YOUR CODE HERE
    raise NotImplementedError


def predict_or_abstain(probs: np.ndarray, threshold: float) -> np.ndarray:
    """Row argmax where the row's top probability is >= `threshold`, ``ABSTAIN`` elsewhere.

    Example:
        >>> predict_or_abstain(np.array([[0.7, 0.3], [0.4, 0.6], [0.55, 0.45]]), 0.6).tolist()
        [0, 1, -1]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_threshold() -> None:
    conf = np.array([0.9, 0.8, 0.8, 0.6])
    corr = np.array([True, True, False, True])
    t, c, a = coverage_accuracy_curve(conf, corr)
    assert np.allclose(t, [0.9, 0.8, 0.6]) and np.allclose(c, [0.25, 0.75, 1.0]), (
        f"expected thresholds [0.9, 0.8, 0.6] at coverage [0.25, 0.75, 1.0], got "
        f"{np.round(t, 3).tolist()} at {np.round(c, 3).tolist()} — the two clauses at 0.8 enter "
        "TOGETHER: one point per distinct confidence, not one per clause")
    assert np.allclose(a, [1.0, 2 / 3, 0.75]), \
        f"accuracy at each point should be [1.0, 0.667, 0.75], got {np.round(a, 3).tolist()}"
    got = choose_threshold(conf, corr, 0.75)
    assert got == 0.6, (
        f"target 0.75 should give threshold 0.6 (coverage 1.0, accuracy exactly 0.75), got "
        f"{got} — the target is inclusive, and the curve dips at 0.8 and recovers: take the "
        "SMALLEST threshold that meets the target, not the last one before the first miss")
    assert choose_threshold(conf, corr, 1.0) == 0.9, "target 1.0 is met only by the top clause"
    assert choose_threshold(conf, np.zeros(4, dtype=bool), 0.5) == math.inf, \
        "when nothing reaches the target, return math.inf so every clause is abstained"
    pred = predict_or_abstain(np.array([[0.6, 0.4], [0.3, 0.7], [0.5, 0.5]]), 0.6)
    assert pred.tolist() == [0, 1, ABSTAIN], (
        f"expected [0, 1, -1], got {pred.tolist()} — a clause EXACTLY at the threshold is "
        "answered (>=), matching how choose_threshold counted it")
    print("exercise 6 looks right")


_try("exercise 6", _check_threshold)

The curve for the calibrated classifier on the calibration and test splits. The dashed line
is your `TARGET_ACCURACY`.

In [ ]:
def _show_curves() -> None:
    temperature = fit_temperature(LOGITS["cal"], Y["cal"])
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    for split in ("cal", "test"):
        P = softmax(LOGITS[split] / temperature)
        _, cov, acc = coverage_accuracy_curve(P.max(axis=1), P.argmax(axis=1) == Y[split])
        ax.plot(cov, acc, label=f"{split} ({len(Y[split])} clauses)")
        for share in (1.0, 0.8, 0.5):
            i = int(np.nonzero(cov >= share - 1e-12)[0][0])
            print(f"{split:5s}: answering the most confident {100 * cov[i]:5.1f}% of clauses, "
                  f"accuracy {acc[i]:.3f}")
    ax.axhline(TARGET_ACCURACY, color="grey", ls="--", lw=0.8, label="TARGET_ACCURACY")
    ax.set_xlabel("coverage (share of clauses answered)")
    ax.set_ylabel("selective accuracy")
    ax.set_xlim(0, 1.01)
    ax.set_ylim(0.85, 1.005)
    ax.set_title("Answer fewer clauses, get more of them right")
    ax.legend(loc="lower left")
    fig.tight_layout()
    _show(fig)


_try("coverage curves", _show_curves, needs=_FOR_MODEL + ("exercise 5", "exercise 6"))

## 9. Exercise 7 — the policy, fitted where it may be, and its output in module 1's format

`fit_policy` is the whole abstention policy in one function: a temperature, then a threshold
on the **calibrated** confidences, for the target you promise. It is handed every split,
because in real code the splits sit side by side in one object and nothing stops a hand
reaching for the wrong one — nothing except a test. `SPLIT_JOBS` says which split each
number may come from. The grader shuffles the labels of every other split and checks that
your policy does not move.

`to_review_records` writes the classifier's output as module 1's records — `doc_id`, `gold`,
`pred`, `conf`, one field called `clause_type` — so module 1's review queue can take it as it
is. An abstention claims nothing, so its prediction is the empty string, exactly as module 1
wrote a field the extractor did not find. Its confidence stays the real calibrated one: the
queue orders the abstained clauses among themselves by it.

<details><summary>💡 Hint 1 — what to think about</summary>

Which split has the job of fitting both numbers, according to SPLIT_JOBS? Is the threshold
compared with raw or calibrated confidences when the policy is applied — and so which kind
must it be chosen on? In the records, look at what `LABELS[-1]` is before you index the label
tuple with a prediction that might be ABSTAIN.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Policy: fit the temperature on the calibration logits and labels; divide the calibration
logits by it, take the softmax, and read each clause's top probability and whether its
argmax is right; hand those to choose_threshold with the target. Records: walk the clauses
in order, write the gold label's name, the predicted label's name or the empty string for
ABSTAIN, and the confidence as a plain float, each under REVIEW_FIELD.

</details>

In [ ]:
REVIEW_FIELD = "clause_type"


def fit_policy(logits: Mapping[str, np.ndarray], labels: Mapping[str, np.ndarray],
               target_accuracy: float) -> Policy:
    """Temperature and threshold for `target_accuracy`, learnt only where SPLIT_JOBS allows.

    `logits` and `labels` map split names to arrays and may hold any of ``SPLITS``. Both the
    temperature (``fit_temperature``) and the threshold (``choose_threshold``, on the
    CALIBRATED top probabilities and whether each argmax is right) come from the split whose
    job is to fit them. Changing any other split's logits or labels must not change the result.

    Example:
        >>> z = {"cal": np.array([[4.0, 0.0], [4.0, 0.0], [4.0, 0.0], [0.0, 4.0]]),
        ...      "test": np.zeros((2, 2))}
        >>> p = fit_policy(z, {"cal": np.array([0, 0, 1, 1]), "test": np.array([0, 1])}, 0.75)
        >>> round(p.temperature, 3), round(p.threshold, 3), p.target_accuracy
        (3.641, 0.75, 0.75)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def to_review_records(clause_ids: Sequence[str], y_true: np.ndarray, y_pred: np.ndarray,
                      confidence: np.ndarray, labels: Sequence[str] = LABELS) -> list[dict]:
    """One module-1 record per clause, in input order, with the single field REVIEW_FIELD.

    ``{"doc_id": id, "gold": {REVIEW_FIELD: true label name},
    "pred": {REVIEW_FIELD: predicted label name, or "" for ABSTAIN},
    "conf": {REVIEW_FIELD: float(confidence)}}`` — the confidence is kept for abstained
    clauses too. Arguments of different lengths raise ``ValueError``.

    Example:
        >>> r = to_review_records(["TE-0001"], np.array([3]), np.array([ABSTAIN]), np.array([0.41]))
        >>> r[0]["pred"], r[0]["gold"], r[0]["conf"]
        ({'clause_type': ''}, {'clause_type': 'assignment'}, {'clause_type': 0.41})
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_policy() -> None:
    rng = np.random.default_rng(5)
    z = {s: rng.normal(size=(n, 3)) * 2.0 for s, n in (("train", 90), ("cal", 120), ("test", 80))}
    y = {s: np.argmax(z[s] + rng.normal(size=z[s].shape) * 1.5, axis=1) for s in z}
    base = fit_policy(z, y, 0.8)
    t_cal = fit_temperature(z["cal"], y["cal"])
    assert abs(math.log(base.temperature) - math.log(t_cal)) < 1e-3, (
        f"the temperature should be fitted on the calibration split ({t_cal:.3f}), got "
        f"{base.temperature:.3f} — SPLIT_JOBS gives that job to 'cal', not to 'train' or 'test'")
    for other in ("train", "test"):
        y2 = dict(y)
        y2[other] = rng.permutation(y[other])
        moved = fit_policy(z, y2, 0.8)
        assert moved == base, (
            f"shuffling the {other!r} labels changed your policy from {base} to {moved} — a "
            f"policy fitted on {other!r} " + ("is fitted on clauses the weights have already "
                                              "memorised" if other == "train" else
                                              "turns the report into a number you chose"))
    P = softmax(z["cal"] / t_cal)
    want = choose_threshold(P.max(1), P.argmax(1) == y["cal"], 0.8)
    assert abs(base.threshold - want) < 1e-9, (
        f"threshold should be {want:.4f}, chosen on the CALIBRATED calibration confidences; "
        f"got {base.threshold:.4f} — did you choose it on the raw softmax?")
    recs = to_review_records(["A", "B"], np.array([5, 0]), np.array([ABSTAIN, 0]),
                             np.array([0.37, 0.93]))
    assert recs[0]["pred"][REVIEW_FIELD] == "", (
        f"an abstention must be written as '' (nothing claimed), got "
        f"{recs[0]['pred'][REVIEW_FIELD]!r} — LABELS[-1] is {LABELS[-1]!r}, which is what "
        "indexing the labels with ABSTAIN silently returns")
    assert recs[0]["conf"][REVIEW_FIELD] == 0.37, \
        "keep the real confidence on an abstained clause; the queue orders abstentions by it"
    assert recs[1] == {"doc_id": "B", "gold": {REVIEW_FIELD: LABELS[0]},
                       "pred": {REVIEW_FIELD: LABELS[0]}, "conf": {REVIEW_FIELD: 0.93}}, \
        f"record shape should match module 1's exactly, got {recs[1]}"
    print("exercise 7 looks right")


_try("exercise 7", _check_policy)

Your policy, at the `TARGET_ACCURACY` you set in the setup cell, reported on the test split
with a bootstrap interval — and what abstaining cost each clause type in recall.

In [ ]:
BOOT_SEED = 20260924
N_BOOT = 2000


def apply_policy(policy: Policy, logits: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(predictions with ABSTAIN, calibrated top probability) for a matrix of logits."""
    P = softmax(np.asarray(logits, dtype=np.float64) / policy.temperature)
    return predict_or_abstain(P, policy.threshold), P.max(axis=1)


def bootstrap_interval(answered: np.ndarray, correct: np.ndarray, n_boot: int = N_BOOT,
                       seed: int = BOOT_SEED) -> tuple[tuple[float, float], tuple[float, float]]:
    """95% percentile bootstrap intervals for (coverage, selective accuracy), resampling
    clauses with replacement. Deterministic for a given seed."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(answered), size=(n_boot, len(answered)))
    a, hit = answered[idx], (answered & correct)[idx]
    cov = a.mean(axis=1)
    acc = hit.sum(axis=1) / np.maximum(a.sum(axis=1), 1)
    q = lambda v: (float(np.quantile(v, 0.025)), float(np.quantile(v, 0.975)))  # noqa: E731
    return q(cov), q(acc)


_FOR_POLICY = _FOR_MODEL + ("exercise 4", "exercise 5", "exercise 6", "exercise 7")


def _show_policy_report() -> None:
    policy = fit_policy(LOGITS, Y, TARGET_ACCURACY)
    print(f"policy for TARGET_ACCURACY = {TARGET_ACCURACY}: T = {policy.temperature:.3f}, "
          f"answer iff calibrated confidence >= {policy.threshold:.3f}\n")
    pred, _ = apply_policy(policy, LOGITS["test"])
    answered = pred != ABSTAIN
    correct = pred == Y["test"]
    (c_lo, c_hi), (a_lo, a_hi) = bootstrap_interval(answered, correct)
    sel = correct[answered].mean() if answered.any() else float("nan")
    print(f"test split, touched once: coverage {answered.mean():.3f} (95% interval {c_lo:.3f} to "
          f"{c_hi:.3f}), selective accuracy {sel:.3f} (95% interval {a_lo:.3f} to {a_hi:.3f})")
    print(f"{int((~answered).sum())} of {len(pred)} test clauses abstained\n")
    full = score_classes(Y["test"], LOGITS["test"].argmax(axis=1))
    held = score_classes(Y["test"], pred)
    print(f"{'clause type':30s}{'answer all':>22s}{'with abstention':>22s}{'recall cost':>13s}")
    print(f"{'':30s}{'prec':>11s}{'recall':>11s}{'prec':>11s}{'recall':>11s}")
    for f, h in zip(full, held):
        print(f"{f.label:30s}{f.precision:11.3f}{f.recall:11.3f}{h.precision:11.3f}"
              f"{h.recall:11.3f}{f.recall - h.recall:13.3f}")
    print(f"{'MACRO F1':30s}{macro_f1(full):22.3f}{macro_f1(held):22.3f}\n")
    for flag in ("hybrid", "smudged", "truncated"):
        mask = np.array([c[flag] for c in CORPUS["test"]])
        print(f"abstention rate on {flag:9s} clauses {(~answered)[mask].mean():.3f}, on the "
              f"rest {(~answered)[~mask].mean():.3f}")


_try("policy report", _show_policy_report, needs=_FOR_POLICY)

## 10. Why the grader checks where the policy was fitted

The threshold is chosen so that accuracy on the tuning clauses **just** reaches the target.
Tune it on the split you then report, and the report is not a measurement any more: it
restates the constraint you imposed. This cell measures the damage. Many times over, it
shuffles every held-out clause (calibration, test and next quarter together) into a tuning
set, a reporting set the size of the test split, and a "future" made of the rest. The honest
policy is tuned on the tuning set and reported on the reporting set; the leaky policy is
tuned on the reporting set itself. The future — clauses neither policy has seen — says what
really happens after the report is signed.

In [ ]:
N_RESPLITS = 100
RESPLIT_SEED = 20260925


def _selective(policy: Policy, logits: np.ndarray, y: np.ndarray) -> float:
    pred, _ = apply_policy(policy, logits)
    answered = pred != ABSTAIN
    return float((pred == y)[answered].mean()) if answered.any() else float("nan")


def _show_why_the_order_matters() -> None:
    held = ("cal", "test", "next_quarter")
    pool_z = np.vstack([LOGITS[s] for s in held])
    pool_y = np.concatenate([Y[s] for s in held])
    n_tune, n_report = len(Y["cal"]), len(Y["test"])
    rng = np.random.default_rng(RESPLIT_SEED)
    results = {"honest": [], "leaky": []}
    for _ in range(N_RESPLITS):
        perm = rng.permutation(len(pool_y))
        tune, report = perm[:n_tune], perm[n_tune:n_tune + n_report]
        future = perm[n_tune + n_report:]
        for name, fit_on in (("honest", tune), ("leaky", report)):
            policy = fit_policy({"cal": pool_z[fit_on]}, {"cal": pool_y[fit_on]}, TARGET_ACCURACY)
            results[name].append((_selective(policy, pool_z[report], pool_y[report]),
                                  _selective(policy, pool_z[future], pool_y[future])))
    print(f"{N_RESPLITS} shuffles of the {len(pool_y)} held-out clauses into {n_tune} to tune, "
          f"{n_report} to report and {len(pool_y) - n_tune - n_report} as the future; target "
          f"{TARGET_ACCURACY}\n")
    print(f"{'policy tuned on':22s}{'report: mean acc':>17s}{'says met':>10s}"
          f"{'future: mean acc':>18s}{'met':>7s}{'optimism':>10s}{'false assurance':>17s}")
    for name, where in (("honest", "the tuning set"), ("leaky", "the reporting set")):
        rep = np.array([r for r, _ in results[name]])
        fut = np.array([f for _, f in results[name]])
        says, met = rep >= TARGET_ACCURACY, fut >= TARGET_ACCURACY
        print(f"{where:22s}{np.nanmean(rep):17.4f}{100 * says.mean():9.0f}%"
              f"{np.nanmean(fut):18.4f}{100 * met.mean():6.0f}%{np.nanmean(rep - fut):+10.4f}"
              f"{100 * (says & ~met).mean():16.0f}%")
    print("\n'optimism': the mean of report minus future. 'false assurance': the report said the")
    print("target was met and the future did not.")


_try("why the order matters", _show_why_the_order_matters,
     needs=_FOR_MODEL + ("exercise 5", "exercise 6", "exercise 7"))

## 11. The abstentions go to module 1's review queue, and the rate gets a price

Module 1 built the queue: records in, the lowest-confidence cells out, ties broken on
`(doc_id, field)`, corrections written into new records. Routing on confidence is the
documented design of a production document service: its documentation tells operators to
use confidence to decide whether to accept a prediction automatically or flag it for human
review. The cell below carries a faithful copy of the pieces this lesson uses — same names,
same record format — because a notebook runs alone and cannot import another lesson. The
four costs are **placeholders** for your own organisation's figures, exactly as in module 1;
nothing here claims they are typical.

In [ ]:
# --- carried from module 1 (P02-L01), unchanged in behaviour, trimmed to what is used ---
SCHEMA: dict[str, str] = {REVIEW_FIELD: "id"}
MATCH_MODES = ("exact", "normalised", "fuzzy")


class FieldScore(NamedTuple):
    """What one field scored over the whole corpus, under one matching mode."""
    field: str
    mode: str
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float


def normalise_value(value: str, field_type: str) -> str:
    """Module 1's rule for ``id`` values (upper case, alphanumerics only); empty in, empty
    out. Only the ``id`` rule is carried: it is the only field type this lesson scores."""
    if not isinstance(value, str) or not value.strip():
        return ""
    if field_type != "id":
        raise ValueError(f"this lesson carries only module 1's 'id' rule, not {field_type!r}")
    raw = " ".join(value.split())
    stripped = "".join(ch for ch in raw if ch.isascii() and ch.isalnum())
    return stripped.upper() if stripped else raw.upper()


def match_value(pred: str, gold: str, field_type: str, mode: str) -> bool:
    """Module 1's matcher. An empty side never matches; fuzzy applies to text only."""
    if mode not in MATCH_MODES:
        raise ValueError(f"unknown mode {mode!r}; expected one of {list(MATCH_MODES)}")
    if not pred.strip() or not gold.strip():
        return False
    if mode == "exact":
        return pred == gold
    return normalise_value(pred, field_type) == normalise_value(gold, field_type)


def score_field(records: Sequence[Mapping], field: str, mode: str) -> FieldScore:
    """Module 1's scorer: a wrong value is a false positive AND a false negative."""
    tp = fp = fn = 0
    for record in records:
        pred, gold = record["pred"][field], record["gold"][field]
        has_pred, has_gold = bool(pred.strip()), bool(gold.strip())
        if has_pred and has_gold and match_value(pred, gold, SCHEMA[field], mode):
            tp += 1
            continue
        if has_pred:
            fp += 1
        if has_gold:
            fn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return FieldScore(field, mode, tp, fp, fn, precision, recall, f1)


def review_queue(records: Sequence[Mapping], budget: int) -> tuple[tuple[str, str], ...]:
    """Module 1's queue: lowest confidence first, ties on (doc_id, field), at most `budget`."""
    if budget < 0:
        raise ValueError(f"budget must be >= 0, got {budget}")
    cells = [(record["conf"][field], record["doc_id"], field)
             for record in records for field in SCHEMA]
    cells.sort(key=lambda cell: (cell[0], cell[1], cell[2]))
    return tuple((doc_id, field) for _, doc_id, field in cells[:budget])


def apply_reviews(records: Sequence[Mapping], routed: Iterable[tuple[str, str]]) -> list[dict]:
    """Module 1's perfect reviewer, writing into NEW records: gold value, confidence 1.0."""
    routed_set = set(routed)
    out = []
    for record in records:
        pred, conf = dict(record["pred"]), dict(record["conf"])
        for field in pred:
            if (record["doc_id"], field) in routed_set:
                pred[field] = record["gold"][field]
                conf[field] = 1.0
        out.append({"doc_id": record["doc_id"], "gold": dict(record["gold"]),
                    "pred": pred, "conf": conf})
    return out


def random_queue(records: Sequence[Mapping], budget: int, seed: int = 11) -> tuple:
    """Module 1's control policy: route `budget` cells chosen at random. Deterministic."""
    cells = [(r["doc_id"], f) for r in records for f in SCHEMA]
    rng = random.Random(seed)
    return tuple(rng.sample(cells, k=min(budget, len(cells))))
# --- end of module 1 ---


SECONDS_PER_REVIEWED_CLAUSE = 90.0   # placeholder: time your own reviewers, do not guess
REVIEWER_COST_PER_HOUR = 30.0        # placeholder: your loaded cost, in your own currency
MACHINE_COST_PER_CLAUSE = 0.002      # placeholder: your inference bill divided by volume
COST_OF_AN_UNREVIEWED_ERROR = 15.0   # placeholder: what one clause filed under the wrong type
                                     # costs you when nobody looks at it
PRICE_TARGETS = (0.95, 0.97, 0.98, 0.99, 0.995, 1.0)


def price_per_clause(abstain_rate: float, unreviewed_error_rate: float) -> tuple[float, float, float]:
    """(review cost, error cost, total) per clause from the placeholder rates above."""
    review = abstain_rate * SECONDS_PER_REVIEWED_CLAUSE / 3600.0 * REVIEWER_COST_PER_HOUR
    errors = unreviewed_error_rate * COST_OF_AN_UNREVIEWED_ERROR
    return review, errors, MACHINE_COST_PER_CLAUSE + review + errors


def _show_review_and_price() -> None:
    policy = fit_policy(LOGITS, Y, TARGET_ACCURACY)
    pred, conf = apply_policy(policy, LOGITS["test"])
    ids = [c["clause_id"] for c in CORPUS["test"]]
    records = to_review_records(ids, Y["test"], pred, conf)
    abstained = {(ids[i], REVIEW_FIELD) for i in np.nonzero(pred == ABSTAIN)[0]}
    queue = review_queue(records, len(abstained))
    print(f"module 1's queue, given a budget equal to the {len(abstained)} abstentions, routes "
          f"exactly the abstained clauses: {set(queue) == abstained}")
    before = score_field(records, REVIEW_FIELD, "normalised")
    after = score_field(apply_reviews(records, queue), REVIEW_FIELD, "normalised")
    control = score_field(apply_reviews(records, random_queue(records, len(abstained))),
                          REVIEW_FIELD, "normalised")
    print(f"module 1's scorer on the test clauses: precision {before.precision:.3f}, recall "
          f"{before.recall:.3f}, F1 {before.f1:.3f} before review")
    print(f"  after reviewing the abstentions      : F1 {after.f1:.3f}")
    print(f"  after reviewing as many at random    : F1 {control.f1:.3f}  (module 1's control)\n")

    print("the price of each promise, on the test split, per 1000 clauses and per clause")
    print(f"{'target':>8s}{'threshold':>11s}{'coverage':>10s}{'abstained':>11s}"
          f"{'unreviewed errors':>19s}{'review':>9s}{'errors':>9s}{'total':>9s}")
    rows = [("all", 0.0)] + [(t, t) for t in sorted(set(PRICE_TARGETS) | {TARGET_ACCURACY})]
    table = []
    for name, target in rows:
        p = Policy(1.0, 0.0, 0.0) if name == "all" else fit_policy(LOGITS, Y, target)
        pr, _ = apply_policy(p, LOGITS["test"])
        abstain_rate = float((pr == ABSTAIN).mean())
        error_rate = float(((pr != ABSTAIN) & (pr != Y["test"])).mean())
        review, errors, total = price_per_clause(abstain_rate, error_rate)
        table.append((name, total))
        mark = "  <- yours" if name == TARGET_ACCURACY else ""
        th = "—" if name == "all" else ("never" if math.isinf(p.threshold) else f"{p.threshold:.3f}")
        print(f"{str(name):>8s}{th:>11s}{1 - abstain_rate:10.3f}{1000 * abstain_rate:11.0f}"
              f"{1000 * error_rate:19.0f}{review:9.3f}{errors:9.3f}{total:9.3f}{mark}")
    cheapest = min(table, key=lambda row: row[1])
    print(f"\ncheapest row under these placeholder costs: target {cheapest[0]} at "
          f"{cheapest[1]:.3f} per clause. Change the four placeholders and it moves; that is "
          "the point.")


_try("review and price", _show_review_and_price,
     needs=_FOR_MODEL + ("exercise 5", "exercise 6", "exercise 7"))

## 12. Common mistakes

- **Fitting the vocabulary, the temperature or the threshold on the test split.** Each one
  turns a number you report into a number you chose. Section 10 measured what that buys.
- **`X[i, ids] += 1`.** Numpy applies a repeated index once. The clause that says "party"
  three times is counted as saying it once, and no error is raised.
- **Indexing with `ABSTAIN`.** `-1` is a legal numpy and Python index: it silently means the
  last clause type. Every abstention becomes a change-of-control prediction.
- **Dropping abstained clauses before scoring.** Recall then never moves, and the cost of
  abstaining disappears from the one table that should show it.
- **Calibrating probabilities instead of logits.** Renormalising `p / T` gives back `p`.
- **Stopping when training "looks done".** A stopping rule is part of the model's
  documentation; write it down and let the loss history prove it was followed.
- **Reading 0.9 as 90% before calibrating.** Section 7 measured how far apart
  the raw and calibrated meanings of 0.9 are on this model.
- **Choosing the threshold at the first dip.** The coverage/accuracy curve is not monotone.
- **Reading the threshold as a guarantee.** It is a point estimate that just meets the target
  on the calibration split. Selective classification with a guaranteed risk (Geifman and
  El-Yaniv, arXiv 1705.08500) chooses it with a high-probability bound instead; section 10
  measured how often the point estimate over-promises.

Two of those raise no error, which is why they survive code review. See them rather than
believe them.

In [ ]:
def _show_two_silent_traps() -> None:
    ids = [1, 1, 1, 3]
    row = np.zeros(4)
    row[ids] += 1
    print(f"row[ids] += 1 with ids {ids}   -> {row.tolist()}   id 1 counted once, not three times")
    row = np.zeros(4)
    np.add.at(row, ids, 1)
    print(f"np.add.at(row, ids, 1)              -> {row.tolist()}")
    print(f"LABELS[ABSTAIN] -> {LABELS[ABSTAIN]!r}: no error, just a clause type nobody predicted")


_show_two_silent_traps()

## 13. Self-check

1. After temperature scaling, the test accuracy printed in section 7 did not change. Why?
   - (a) the temperature was fitted on a different split, so it cannot help the test split
   - (b) dividing every logit of a clause by the same positive T cannot change which type
         has the largest logit
   - (c) the model was already calibrated

2. Section 7 printed the fitted temperature. If T is below 1, the raw classifier was:
   - (a) underconfident — its top probabilities were lower than its accuracy justified
   - (b) overconfident — its top probabilities were higher than its accuracy justified
   - (c) badly trained, and should be retrained with a smaller penalty before calibrating

3. In section 10, read the leaky row's "says met" column against its "met" column for the
   future. The right reading is:
   - (a) tuning on the reported split is more data-efficient, so it should be preferred
   - (b) the future happened to be harder than the reporting set
   - (c) the threshold was chosen to make that report meet the target, so "met" is the
         constraint restated, not evidence about new clauses

4. You abstain on some clauses at your target. For `change_of_control`, the effect on its
   precision and recall is:
   - (a) recall can only fall or hold, because an abstained clause is a false negative for its
         true type and a false positive for none; precision usually rises
   - (b) both can only rise, because the abstained clauses were the uncertain ones
   - (c) neither moves, because abstained clauses are excluded from scoring

5. A colleague proposes "answer when the model's probability is at least 0.9" on the raw,
   uncalibrated classifier, to save the calibration step. The strongest objection is:
   - (a) 0.9 is an arbitrary round number
   - (b) on an uncalibrated model 0.9 does not mean "right nine times in ten", so neither the
         coverage nor the accuracy that rule delivers is what its author thinks it is
   - (c) raw probabilities cannot be compared with a threshold at all

Answers come with this lesson's worked solution when you enrol on Synapsa.

One last cell: the scorecard. Every line computed, which is the shape of the summary a
model-risk reviewer will ask you for.

In [ ]:
def _show_scorecard() -> None:
    policy = fit_policy(LOGITS, Y, TARGET_ACCURACY)
    pred, _ = apply_policy(policy, LOGITS["test"])
    answered = pred != ABSTAIN
    full = score_classes(Y["test"], LOGITS["test"].argmax(axis=1))
    held = score_classes(Y["test"], pred)
    worst = max(zip(full, held), key=lambda fh: fh[0].recall - fh[1].recall)
    raw = softmax(LOGITS["test"])
    cal = softmax(LOGITS["test"] / policy.temperature)
    review, errors, total = price_per_clause(float((~answered).mean()),
                                             float((answered & (pred != Y["test"])).mean()))
    print(f"vocabulary        {TOKENIZER.get_vocab_size()} BPE subwords from the training split, "
          f"fingerprint {FINGERPRINT[:16]}")
    rule = "the TOL rule" if MODEL.n_iter < MAX_ITER else "MAX_ITER"
    print(f"classifier        softmax regression, {MODEL.n_iter} full-batch steps, stopped by {rule}")
    print(f"answering all     test accuracy {(raw.argmax(1) == Y['test']).mean():.3f}, macro F1 "
          f"{macro_f1(full):.3f}")
    print(f"calibration       T = {policy.temperature:.3f} from the calibration split; test ECE "
          f"{expected_calibration_error(raw, Y['test']):.3f} -> "
          f"{expected_calibration_error(cal, Y['test']):.3f}")
    print(f"abstention        target {TARGET_ACCURACY}, threshold {policy.threshold:.3f}: "
          f"coverage {answered.mean():.3f}, selective accuracy "
          f"{(pred == Y['test'])[answered].mean():.3f} on the test split")
    print(f"recall cost       largest on {worst[0].label} ({worst[0].recall:.3f} -> "
          f"{worst[1].recall:.3f})")
    print(f"price             {total:.3f} per clause in the placeholder cost model "
          f"({review:.3f} review, {errors:.3f} unreviewed errors)")


_try("scorecard", _show_scorecard, needs=_FOR_POLICY)

## What you built, and where it goes next

A clause classifier that says "I don't know" at a rate you chose and priced: a vocabulary you
proved reproducible, a model with a written stopping rule, probabilities that mean what they
say, a threshold fitted where it may be, and abstentions that land in module 1's review queue
as ordinary records. Module 6 takes away module 1's perfect reviewer; module 9 turns this
section's price table into a routing budget.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_bag),
                             ("exercise 2", _check_softmax_and_loss),
                             ("exercise 3", _check_training),
                             ("exercise 4", _check_scores),
                             ("exercise 5", _check_calibration),
                             ("exercise 6", _check_threshold),
                             ("exercise 7", _check_policy)):
            _try(_name, _check)
    _progress_board()
    _wall = time.perf_counter() - _LESSON_T0
    # Whole seconds: two machines disagree at the first decimal, and that is noise, not a result.
    print("\nlesson wall time so far: " + ("under a second" if _wall < 1 else f"{_wall:.0f}s"))
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.